# LLM-Compiler (parallel DAG)

> Notebook generated from [`examples/patterns/04_llm_compiler.py`](../../examples/patterns/04_llm_compiler.py).

| Field | Value |
|------|-------|
| **Dataset** | HotpotQA (embedded multi-hop) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** DAG validated by Kahn (topological list); wave-based execution with per-task parallel timings; synthesized answer.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
LLM-Compiler — Generation of research reports as a parallel DAG
=========================================================================
Pattern: SPEC-PAT-005 / prismal.agents.patterns.llm_compiler

Dataset: HotpotQA (multi-hop reasoning questions)
  • 113,000 questions that require reasoning over multiple documents.
  • Reference: https://huggingface.co/datasets/hotpot_qa
  • Why: LLMCompiler decomposes complex goals into a DAG of tasks that
    execute in parallel. HotpotQA demands exactly this kind of
    distributed reasoning (search A and B, then synthesize with A+B).

Pattern description:
  LLMCompiler converts a goal into a DAG of tasks (TaskNode), validates
  the graph with Kahn's algorithm, computes topological execution waves
  and runs each wave concurrently with asyncio.gather.
  If a task fails, it re-plans up to max_replanning times.

Required callables:
  - plan_fn(goal, state, previous_results) → list of TaskNode
  - tool_executor(task, prior_outputs) → task output
  - joiner(goal, completed_tasks) → final answer

Usage:
    uv run python examples/patterns/04_llm_compiler.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
from typing import Any

from prismal.agents.patterns.llm_compiler import (
    CompilerResult,
    LLMCompiler,
    TaskNode,
)

## Dataset: HotpotQA questions

In [ ]:
# Representative subset that requires multi-hop reasoning.
HOTPOTQA_SAMPLES = [
    {
        "id": "Q1",
        "question": (
            "What is the capital of the country where the Python programming "
            "language was invented, and how many inhabitants does that city have?"
        ),
        "expected_subtasks": ["find_python_origin", "find_capital", "find_population"],
    },
    {
        "id": "Q2",
        "question": (
            "Who founded the company that developed the GPT-4 model, and in what "
            "year was that company founded?"
        ),
        "expected_subtasks": ["find_gpt4_company", "find_founder", "find_founding_year"],
    },
    {
        "id": "Q3",
        "question": (
            "Compare LangChain and LlamaIndex: which has more stars on GitHub "
            "and which has greater adoption in Fortune 500 companies?"
        ),
        "expected_subtasks": [
            "find_langchain_github",
            "find_llamaindex_github",
            "find_enterprise_adoption",
            "compare_frameworks",
        ],
    },
]

## Simulation of research tools

In [ ]:
# In production these tools would call real APIs (web search, Wikipedia, etc.)

_MOCK_TOOL_RESULTS: dict[str, str] = {
    "search": "Search results: relevant information found",
    "wikipedia": "Wikipedia: article with detailed information",
    "web_scrape": "Web page: extracted structured data",
    "aggregate": "Aggregation: synthesis from multiple sources",
    "compare": "Comparison: side-by-side analysis completed",
    "calculate": "Calculation: numerical result obtained",
}

## Compiler callables

In [ ]:
async def plan_fn(
    goal: str,
    state: Any,
    previous_results: dict[str, Any] | None = None,
) -> list[TaskNode]:
    """Decompose a goal into a DAG of tasks.

    In production, an LLM would generate the plan. Here we use keyword-based
    heuristics to create representative plans.

    Args:
        goal: Goal to achieve.
        state: Current agent state.
        previous_results: Results from a previous failed plan (for replanning).

    Returns:
        List of TaskNode forming a valid DAG.
    """
    # Replanning: if there were previous failures, add verification tasks
    extra_verify = []
    if previous_results:
        failed = [k for k, v in previous_results.items() if v is None]
        if failed:
            extra_verify = [
                TaskNode(
                    id="T_verify",
                    description=f"Verify and recover failed tasks: {failed}",
                    tool="web_scrape",
                    args={"query": f"alternative for {failed}"},
                    depends_on=[],
                )
            ]

    # Heuristic plan based on the question
    if "python" in goal.lower() or "language" in goal.lower():
        tasks = [
            TaskNode(
                id="T1",
                description="Find the origin of the Python language",
                tool="search",
                args={"query": "Python programming language origin country creator"},
                depends_on=[],
            ),
            TaskNode(
                id="T2",
                description="Identify the country and its capital",
                tool="wikipedia",
                args={"query": "Netherlands capital Amsterdam", "context": "$T1.output"},
                depends_on=["T1"],
            ),
            TaskNode(
                id="T3",
                description="Find the population of the capital",
                tool="search",
                args={"query": "Amsterdam population 2024", "context": "$T2.output"},
                depends_on=["T2"],
            ),
        ]
    elif "gpt" in goal.lower() or "openai" in goal.lower():
        tasks = [
            TaskNode(
                id="T1",
                description="Find which company developed GPT-4",
                tool="search",
                args={"query": "GPT-4 developer company"},
                depends_on=[],
            ),
            TaskNode(
                id="T2",
                description="Find the founder of OpenAI",
                tool="wikipedia",
                args={"query": "OpenAI founders", "context": "$T1.output"},
                depends_on=["T1"],
            ),
            TaskNode(
                id="T3",
                description="Find the founding year",
                tool="search",
                args={"query": "OpenAI founding year", "context": "$T1.output"},
                depends_on=["T1"],
            ),
            TaskNode(
                id="T4",
                description="Aggregate founder and year",
                tool="aggregate",
                args={"sources": ["$T2.output", "$T3.output"]},
                depends_on=["T2", "T3"],
            ),
        ]
    else:
        # Generic plan for comparisons
        tasks = [
            TaskNode(
                id="T1",
                description="Gather LangChain data from GitHub",
                tool="web_scrape",
                args={"url": "https://github.com/langchain-ai/langchain"},
                depends_on=[],
            ),
            TaskNode(
                id="T2",
                description="Gather LlamaIndex data from GitHub",
                tool="web_scrape",
                args={"url": "https://github.com/run-llama/llama_index"},
                depends_on=[],
            ),
            TaskNode(
                id="T3",
                description="Find enterprise adoption of both frameworks",
                tool="search",
                args={"query": "LangChain vs LlamaIndex enterprise adoption Fortune 500"},
                depends_on=[],
            ),
            TaskNode(
                id="T4",
                description="Compare the three results",
                tool="compare",
                args={
                    "langchain_data": "$T1.output",
                    "llamaindex_data": "$T2.output",
                    "enterprise_data": "$T3.output",
                },
                depends_on=["T1", "T2", "T3"],
            ),
        ]

    return extra_verify + tasks


async def tool_executor(
    task: TaskNode,
    prior_outputs: dict[str, Any],
) -> str:
    """Execute a task and return its output.

    In production it would invoke real tools (web search, APIs, etc.).
    Here it simulates execution with realistic mock results.

    Args:
        task: The task to execute.
        prior_outputs: Outputs of prior tasks (by ID).

    Returns:
        Task output as a string.
    """
    # Simulate tool latency (real operations would take longer)
    await asyncio.sleep(0.05)

    tool_result = _MOCK_TOOL_RESULTS.get(task.tool, "result not available")

    # Enrich with dependency context
    if prior_outputs:
        context_ids = ", ".join(prior_outputs.keys())
        return f"[{task.tool}] {task.description} → {tool_result} (using context: {context_ids})"

    return f"[{task.tool}] {task.description} → {tool_result}"


async def joiner(goal: str, completed_tasks: list[TaskNode]) -> str:
    """Synthesize the results of all tasks into the final answer.

    Args:
        goal: Original goal.
        completed_tasks: All completed tasks with their outputs.

    Returns:
        Final synthesized answer.
    """
    outputs = [
        f"  • [{t.id}] {t.description}: {str(t.output)[:100]}"
        for t in completed_tasks
        if t.output is not None
    ]
    outputs_str = "\n".join(outputs)

    return (
        f"Synthesized answer for: '{goal}'\n"
        f"Based on {len(completed_tasks)} completed tasks:\n"
        f"{outputs_str}"
    )

## Main function

In [ ]:
async def run_compiler(sample: dict) -> CompilerResult:
    """Run the LLMCompiler for a HotpotQA question.

    Args:
        sample: Question with metadata.

    Returns:
        CompilerResult with the executed plan and the final answer.
    """
    compiler = LLMCompiler(
        tools=[{"name": t} for t in _MOCK_TOOL_RESULTS],
        plan_fn=plan_fn,
        tool_executor=tool_executor,
        joiner=joiner,
        max_replanning=2,
    )

    return await compiler.compile_and_run(
        goal=sample["question"],
        state={"messages": [], "metadata": {}},
    )


async def main() -> None:
    print("=" * 70)
    print("  LLM-Compiler — Dataset: HotpotQA (multi-hop reasoning)")
    print("=" * 70)

    for sample in HOTPOTQA_SAMPLES:
        print(f"\n[{sample['id']}] {sample['question']}")
        print(f"  Expected subtasks: {sample['expected_subtasks']}")

        result = await run_compiler(sample)

        # Show the plan as a DAG
        print(f"\n  DAG plan ({len(result.plan.tasks)} tasks):")
        for t in result.plan.tasks:
            deps = f" → depends on {t.depends_on}" if t.depends_on else " (root)"
            print(f"    {t.id}: [{t.tool}] {t.description}{deps}")

        # Show execution waves (concurrency)
        waves = result.plan.execution_waves
        print(f"\n  Execution waves ({len(waves)} parallel waves):")
        for i, wave in enumerate(waves, 1):
            print(f"    Wave {i}: {wave}  ← executed simultaneously")

        # Results
        print("\n  Task status:")
        for t in result.plan.tasks:
            status_icon = "✓" if t.status == "completed" else "✗"
            print(f"    {status_icon} {t.id} [{t.status}]")

        print(f"\n  Re-plannings: {result.replanning_count}")
        print("\n  Final answer:")
        print(f"  {result.final_answer[:300]}")
        print("-" * 70)

    # Demonstrate DAG validation
    print("\n[DAG validation with Kahn's algorithm]")
    compiler = LLMCompiler(
        tools=[],
        plan_fn=plan_fn,
        tool_executor=tool_executor,
        joiner=joiner,
    )
    valid_plan_tasks = [
        TaskNode(id="A", description="task A", tool="search", args={}, depends_on=[]),
        TaskNode(id="B", description="task B", tool="search", args={}, depends_on=["A"]),
        TaskNode(id="C", description="task C", tool="search", args={}, depends_on=["A"]),
        TaskNode(id="D", description="task D", tool="aggregate", args={}, depends_on=["B", "C"]),
    ]
    from prismal.agents.patterns.llm_compiler import CompilerPlan

    plan = CompilerPlan(tasks=valid_plan_tasks, goal="test", is_valid=False, execution_waves=[])
    is_valid = compiler.validate_dag(plan)
    waves = compiler.compute_execution_waves(valid_plan_tasks)
    print(f"  DAG A→B,C→D  is valid: {is_valid}")
    print(f"  Waves: {waves}")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()